# Option 4: Transfer Learning (MobileNetV2)

Instead of training a convolutional network from scratch (Option 3), this notebook reuses a **MobileNetV2** backbone pretrained on ImageNet. The pretrained convolutional features already know how to detect edges, textures, and shapes, so we only need to teach a small classification head to map those features to our 3 animal classes (`cats`, `dogs`, `panda`). This almost always beats a from-scratch CNN on a small dataset like ours (~2,400 training images).

The workflow has two phases, the standard transfer-learning recipe:

1. **Feature extraction** — freeze the entire MobileNetV2 backbone and train only the new `Dense` head with a normal learning rate (`1e-3`). This is fast and gets the head into a good region without disturbing the pretrained weights.
2. **Fine-tuning** — unfreeze the top block of the backbone and continue training with a very small learning rate (`1e-5`) so the pretrained features gently adapt to our data without being destroyed. BatchNorm layers are kept frozen during this phase.

The data pipeline, one-hot labels, softmax output, and `val_loss`-based checkpointing/early stopping match the earlier notebooks. Note: MobileNetV2 expects inputs scaled to `[-1, 1]`, which the model applies internally via a `Rescaling` layer, so images are fed in as raw `0..255` values.


In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from pathlib import Path


In [ ]:
image_size = (128, 128)
batch_size = 32
seed = 42

# Resolve the dataset path whether the notebook runs from the repo root or Assignment_2.
dataset_dir = Path("Assignment_2/ImageDataset/images")
if not dataset_dir.exists():
    dataset_dir = Path("ImageDataset/images")

image_dataset = tf.keras.utils.image_dataset_from_directory(
    str(dataset_dir),
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed,
)

class_names = image_dataset.class_names
image_batches = []
label_batches = []

# Keep images and labels from the same batch so they stay correctly matched.
for images, labels in image_dataset:
    image_batches.append(images.numpy())
    label_batches.append(labels.numpy())

x_data = np.concatenate(image_batches, axis=0)
y_data = np.concatenate(label_batches, axis=0)

train_images, test_images, train_labels, test_labels = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=seed,
    stratify=y_data,
)

(x_train, y_train), (x_test, y_test) = (train_images, train_labels), (test_images, test_labels)

num_classes = len(class_names)

# Keras applications expect raw 0..255 inputs; the model applies MobileNetV2's
# own preprocessing (scaling to [-1, 1]) as a Rescaling layer, so we do NOT
# divide by 255 here.
image_train = x_train.astype("float32")
image_test = x_test.astype("float32")

# categorical_crossentropy expects one-hot encoded labels rather than integer class ids.
labels_train = to_categorical(y_train, num_classes=num_classes)
labels_test = to_categorical(y_test, num_classes=num_classes)

print(class_names)
print(image_train.shape, labels_train.shape)
print(image_test.shape, labels_test.shape)
print(image_train.dtype, image_train.min(), image_train.max())


In [ ]:
input_shape = image_train.shape[1:]

# Light, label-preserving augmentation, applied before the backbone so the model
# still generalizes even with the small dataset.
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

# MobileNetV2 pretrained on ImageNet, without its 1000-class classification head.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=input_shape,
    include_top=False,
    weights="imagenet",
)
# Phase 1: freeze the whole backbone and train only the new classifier head.
base_model.trainable = False

inputs = Input(shape=input_shape, name="input_image")
x = data_augmentation(inputs)
# MobileNetV2 expects inputs scaled to [-1, 1]; this Rescaling is exactly
# equivalent to tf.keras.applications.mobilenet_v2.preprocess_input.
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_preprocess")(x)
# training=False keeps the frozen BatchNorm layers in inference mode.
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.2, name="head_dropout")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)

model = Model(inputs=inputs, outputs=outputs, name="animal_transfer_mobilenetv2")

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()
print(f"Total layers in base model: {len(base_model.layers)}")
print(f"Trainable base-model layers (phase 1): {sum(layer.trainable for layer in base_model.layers)}")


In [ ]:
# Phase 1 — feature extraction: train only the classifier head on top of the
# frozen MobileNetV2 backbone.
checkpoint_dir = Path("Assignment_2")
if not checkpoint_dir.exists():
    checkpoint_dir = Path(".")

best_model_path = checkpoint_dir / "animal_transfer_mobilenetv2_best.keras"

feature_extraction_epochs = 15

# Reused across both phases: EarlyStopping resets its state each fit, while
# ModelCheckpoint keeps the global best val_loss, so the single saved file is
# always the best model across the whole run.
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=5,
        restore_best_weights=True,
    ),
]

history_head = model.fit(
    image_train,
    labels_train,
    validation_data=(image_test, labels_test),
    epochs=feature_extraction_epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Phase 2 — fine-tuning: unfreeze the top block of the backbone and continue
# training with a very small learning rate so the pretrained features adapt to
# our classes without being washed out.
base_model.trainable = True

# Keep the earliest (most generic) layers frozen; only fine-tune the top ~30.
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# BatchNorm layers must stay in inference mode during fine-tuning, otherwise
# their running statistics get corrupted by the tiny batches.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile (required after changing trainability) with a much smaller LR.
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"],
)

print(f"Trainable base-model layers (phase 2): {sum(layer.trainable for layer in base_model.layers)}")

fine_tune_epochs = 15
total_epochs = feature_extraction_epochs + fine_tune_epochs

history_finetune = model.fit(
    image_train,
    labels_train,
    validation_data=(image_test, labels_test),
    epochs=total_epochs,
    initial_epoch=len(history_head.epoch),
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
import matplotlib.pyplot as plt

# Stitch the two training phases into single continuous curves.
acc = history_head.history["accuracy"] + history_finetune.history["accuracy"]
val_acc = history_head.history["val_accuracy"] + history_finetune.history["val_accuracy"]
loss = history_head.history["loss"] + history_finetune.history["loss"]
val_loss = history_head.history["val_loss"] + history_finetune.history["val_loss"]

epoch_range = range(1, len(acc) + 1)
fine_tune_start = len(history_head.history["accuracy"]) + 0.5

fig, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize=(14, 5))

loss_axis.plot(epoch_range, loss, label="Training loss")
loss_axis.plot(epoch_range, val_loss, label="Validation loss")
loss_axis.axvline(fine_tune_start, color="gray", linestyle="--", linewidth=1, label="Fine-tuning starts")
loss_axis.set_title("Loss over epochs")
loss_axis.set_xlabel("Epoch")
loss_axis.set_ylabel("Loss")
loss_axis.legend()
loss_axis.grid(True, alpha=0.3)

accuracy_axis.plot(epoch_range, acc, label="Training accuracy")
accuracy_axis.plot(epoch_range, val_acc, label="Validation accuracy")
accuracy_axis.axvline(fine_tune_start, color="gray", linestyle="--", linewidth=1, label="Fine-tuning starts")
accuracy_axis.set_title("Accuracy over epochs")
accuracy_axis.set_xlabel("Epoch")
accuracy_axis.set_ylabel("Accuracy")
accuracy_axis.legend()
accuracy_axis.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = int(np.argmin(val_loss)) + 1
gap = max(acc) - max(val_acc)
print(f"Lowest validation loss:   {min(val_loss):.4f} (epoch {best_epoch})")
print(f"Best training accuracy:   {max(acc):.4f}")
print(f"Best validation accuracy: {max(val_acc):.4f}")
print(f"Train/val accuracy gap:   {gap:.4f}  (smaller is better)")


## What to look for

- **High accuracy, fast.** Because MobileNetV2 already learned rich visual features on ImageNet, validation accuracy should jump well above the from-scratch results within just a few feature-extraction epochs — typically into the 0.90+ range for cats/dogs/panda.
- **No BatchNorm warm-up plateau.** Unlike the from-scratch model, validation accuracy should be high from epoch 1 (the backbone's BN statistics are already trained), so you should not see the 0.3333 random-guess plateau.
- **Fine-tuning bump.** After the dashed "Fine-tuning starts" line, the small-LR phase usually gives a further, gentler improvement. If validation loss instead starts rising there, the fine-tuning LR is too high or too many layers were unfrozen.
- **Smaller overfitting gap.** The train/val gap is usually smaller than the from-scratch models because the pretrained features generalize well.


In [ ]:
model_dir = Path("Assignment_2")
if not model_dir.exists():
    model_dir = Path(".")

model_path = model_dir / "animal_transfer_mobilenetv2.keras"
weights_path = model_dir / "animal_transfer_mobilenetv2.weights.h5"
labels_path = model_dir / "class_names_transfer_mobilenetv2.json"

model.save(model_path)
model.save_weights(weights_path)

with open(labels_path, "w", encoding="utf-8") as labels_file:
    json.dump(class_names, labels_file, indent=2)

print(f"Saved restored best model to: {model_path.resolve()}")
print(f"Saved checkpointed best model to: {best_model_path.resolve()}")
print(f"Saved weights to: {weights_path.resolve()}")
print(f"Saved class labels to: {labels_path.resolve()}")


In [ ]:
import matplotlib.pyplot as plt

loaded_model = tf.keras.models.load_model(model_path)

with open(labels_path, "r", encoding="utf-8") as labels_file:
    saved_class_names = json.load(labels_file)

loaded_loss, loaded_accuracy = loaded_model.evaluate(image_test, labels_test, verbose=0)
print(f"Loaded model test loss: {loaded_loss:.4f}")
print(f"Loaded model test accuracy: {loaded_accuracy:.4f}")

sample_count = min(9, len(image_test))
sample_indices = np.arange(sample_count)
predictions = loaded_model.predict(image_test[sample_indices], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(labels_test, axis=1)

fig, axes = plt.subplots(3, 3, figsize=(10, 9), constrained_layout=True)
for axis, image_index, prediction_index in zip(axes.ravel(), sample_indices, range(sample_count)):
    true_label = saved_class_names[true_labels[image_index]]
    predicted_label = saved_class_names[predicted_labels[prediction_index]]
    confidence = predictions[prediction_index][predicted_labels[prediction_index]]
    title_color = "green" if true_label == predicted_label else "red"

    axis.imshow(x_test[image_index].astype("uint8"))
    axis.set_title(
        f"True: {true_label}\nPred: {predicted_label} ({confidence:.2f})",
        color=title_color,
        fontsize=10,
    )
    axis.axis("off")

plt.show()
